# Lab 2 — Colour Spaces & Image Formats

**BITS F459 · Computer Vision · Week 2 · 10 marks**

In the demonstration you saw three things happen to one photograph:

1. Shrinking the two colour channels was almost invisible. Shrinking the brightness
   channel was obvious — even though the first threw away twice as much data.
2. The same photograph saved as a JPEG was about ten times smaller than as a PNG, and
   looked the same.
3. Saving a JPEG twenty times over slowly damaged it. Saving a PNG twenty times changed
   nothing at all.

Each of those is true for that photograph. **None of them is true for every photograph.**

Today you write the conversions yourself, use them to work out what was done to a damaged
picture, and then go looking for the pictures where those three statements stop being
true.

| Part | | Marks |
|:--|:--|:--:|
| **A** | Write the conversions, then work out what was done to a damaged picture | 4 |
| **B** | Find a picture where statement 1 fails, then one where statement 2 fails | 4 |
| **C** | Make the smallest file of the class photo — leaderboard | 2 |

Your BITS ID seeds your test pixel and your mystery image. Your own pictures drive
everything else, so your numbers are yours alone.

## 0 · Setup

## The toolbox — what we import, and why

All of these are already installed in Colab. Nothing to `pip install`.

| Library | What it is | What we use it for today |
|:--|:--|:--|
| **NumPy** (`np`) | fast arrays of numbers | A picture **is** a NumPy array — height × width × 3. Every conversion, average and subtraction today is arithmetic on that array |
| **Pillow** (`PIL.Image`) | reading and writing image files | This is what actually *makes the file*: it does the JPEG, PNG and WebP encoding. Whenever we talk about file size, Pillow produced those bytes |
| **Matplotlib** (`plt`) | plotting | Showing pictures and channels side by side inside the notebook |
| **OpenCV** (`cv2`) | the standard computer-vision library | Used **only as the answer key** — you write the colour conversions yourself, and we check yours against OpenCV's |
| `io` | in-memory files | Lets us "save" an image to memory and measure its size without writing to disk |
| `hashlib` | turns any text into a fixed jumble of digits | Turns your BITS ID into your own test pixel and your own mystery change |

Plumbing you can ignore: `base64`, `json`, `zlib`, `urllib`, `datetime` — used to package your
results at the end.

In [ ]:
import base64, hashlib, io, json, zlib, urllib.request
from datetime import datetime, timezone

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

try:
    import cv2
except ImportError:
    !pip install -q opencv-python-headless
    import cv2

np.set_printoptions(precision=3, suppress=True)
print("numpy", np.__version__, "| opencv", cv2.__version__, "| Pillow", Image.__version__)

In [ ]:
BITS_ID = ""          # ←←← YOUR BITS ID, e.g. "2023A7PS1234U"

assert BITS_ID.strip(), "Put your BITS ID above before running anything else."
SEED = hashlib.sha256(BITS_ID.strip().upper().encode()).hexdigest()[:8]
_s = int(SEED, 16)
PIX_R = 30 + (_s % 200)
PIX_G = 30 + ((_s >> 8) % 200)
PIX_B = 30 + ((_s >> 16) % 200)
if PIX_R == PIX_G == PIX_B:
    PIX_R = 30 + ((PIX_R + 61) % 200)

print(f"  BITS ID     {BITS_ID.strip().upper()}")
print(f"  lab seed    {SEED}")
print(f"  YOUR PIXEL  R={PIX_R}  G={PIX_G}  B={PIX_B}")

---

# Part A — 4 marks

## A1 · One pixel, by hand — 1 mark

The cell above printed three numbers: your own R, G and B. They come from your BITS ID, so
they are yours alone.

Take those three numbers and write them three other ways, **on paper with a calculator**:

- **Greyscale** — one number, the brightness of that pixel.
- **YCrCb** — brightness `Y`, plus `Cr` (how far the pixel leans red) and `Cb` (how far it
  leans blue). The value 128 in `Cr` or `Cb` means no lean at all.
- **HSV** — `H` the hue, an angle from 0° to 360° saying which colour it is; `S` the
  saturation, from 0 (grey) to 1 (fully coloured); `V` the value, from 0 (black) to 1
  (brightest).

The formulas:

$$Y = 0.299R + 0.587G + 0.114B \qquad C_r = 128 + 0.713(R-Y) \qquad C_b = 128 + 0.564(B-Y)$$

For HSV, first divide all three by 255 so they run from 0 to 1. Then $C_{max}$ is simply
the largest of the three, $C_{min}$ the smallest, and $\Delta$ the gap between them:

$$V = C_{max} \qquad S = \Delta / C_{max} \qquad
H = \begin{cases}
60°\left(\frac{g-b}{\Delta} \bmod 6\right) & C_{max}=r \\
60°\left(\frac{b-r}{\Delta} + 2\right) & C_{max}=g \\
60°\left(\frac{r-g}{\Delta} + 4\right) & C_{max}=b
\end{cases}$$

Which of the three lines for $H$ you use depends on which channel was the largest.

Fill in the cell below as you go. **Write down every step, not only the final answers** —
the working is what shows you did it by hand.

**MY WORKING** *(double-click to edit)*

R = `___`  G = `___`  B = `___`

| Step | Value |
|:--|:--|
| $Y = 0.299(\_) + 0.587(\_) + 0.114(\_)$ | `___` |
| $R-Y$, then $C_r$ | `___` → `___` |
| $B-Y$, then $C_b$ | `___` → `___` |
| $r, g, b$ | `___`, `___`, `___` |
| $C_{max}$ (and which channel) | `___` (`___`) |
| $C_{min}$, $\Delta$ | `___`, `___` |
| Which $H$ case applies, and why | `___` |
| $H$, $S$, $V$ | `___`, `___`, `___` |

In [ ]:
A = {"gray": None, "Cr": None, "Cb": None,
     "H": None, "S": None, "V": None, "max_channel": None}
assert all(v is not None for v in A.values()), "Fill in every entry."

def _truth(R, G, B):
    R, G, B = float(R), float(G), float(B)
    Y = .299*R + .587*G + .114*B
    r, g, b = R/255, G/255, B/255
    cmax, cmin = max(r, g, b), min(r, g, b); d = cmax - cmin
    h = 0. if d == 0 else (60*(((g-b)/d) % 6) if cmax == r else
                           60*(((b-r)/d)+2) if cmax == g else 60*(((r-g)/d)+4))
    return {"gray": Y, "Cr": 128+.713*(R-Y), "Cb": 128+.564*(B-Y),
            "H": h, "S": (d/cmax if cmax else 0.), "V": cmax,
            "max_channel": "RGB"[int(np.argmax([R, G, B]))]}

_T = _truth(PIX_R, PIX_G, PIX_B)
_tol = {"gray": .5, "Cr": .5, "Cb": .5, "H": 1., "S": .01, "V": .01}
_ok = all((str(A[k]).strip().upper() == _T[k]) if k == "max_channel"
          else abs(float(A[k]) - _T[k]) <= _tol[k] for k in A)
for k in A:
    good = (str(A[k]).strip().upper() == _T[k]) if k == "max_channel" else abs(float(A[k])-_T[k]) <= _tol[k]
    if good:                 hint = ""
    elif k == "max_channel": hint = "<- not that channel"
    else:                    hint = "<- too high" if float(A[k]) > _T[k] else "<- too low"
    print(f"  {'OK ' if good else 'NO '} {k:12s} you: {str(A[k]):>10s}   {hint}")
A1_ok = _ok
print(f"\nA1: {'PASS (1 mark)' if _ok else 'FAIL'}")

## A2 · The same four conversions, in code — 1 mark

You just did one pixel by hand. Now write the same conversions so they work on a whole
picture at once — no loops needed, NumPy applies the arithmetic to every pixel for you.

Four functions to complete:

| Function | You give it | It returns |
|:--|:--|:--|
| `rgb_to_gray` | a picture, height × width × 3 | one brightness value per pixel, height × width |
| `rgb_to_ycrcb` | a picture | height × width × 3, stacked in the order **Y, Cr, Cb** |
| `ycrcb_to_rgb` | a Y, Cr, Cb picture | back to R, G, B |
| `rgb_to_hsv` | a picture | height × width × 3, stacked as **H, S, V** |

The cell after them checks your work against OpenCV, which already has these conversions
built in. **Do not call `cv2.cvtColor` inside your functions** — that would be checking
OpenCV against itself.

Two things trip almost everyone up:

- OpenCV stores the order as **Y, Cr, Cb** — Cr comes before Cb. Stack yours the same way.
- OpenCV squeezes 8-bit HSV into `H` 0–179 and `S`, `V` 0–255. **Yours should use the
  ordinary units**: `H` from 0 to 360, `S` and `V` from 0 to 1. The check converts
  OpenCV's numbers to match yours, so do not adjust yours to match OpenCV.

In [ ]:
def rgb_to_gray(im):
    a = np.asarray(im, np.float64)
    raise NotImplementedError("Gray = 0.299R + 0.587G + 0.114B")

def rgb_to_ycrcb(im):
    a = np.asarray(im, np.float64)
    raise NotImplementedError("stack (Y, Cr, Cb) - Cr before Cb")

def ycrcb_to_rgb(ycc):
    y = np.asarray(ycc, np.float64); Y, Cr, Cb = y[..., 0], y[..., 1], y[..., 2]
    raise NotImplementedError("R = Y+1.403(Cr-128); G = Y-0.714(Cr-128)-0.344(Cb-128); B = Y+1.773(Cb-128)")

def rgb_to_hsv(im):
    a = np.asarray(im, np.float64)/255.
    r, g, b = a[..., 0], a[..., 1], a[..., 2]
    cmax, cmin = a.max(-1), a.min(-1); d = cmax - cmin
    raise NotImplementedError("three hue cases; guard d == 0")

In [ ]:
# ── check (given) ──
_p = np.array([[[PIX_R, PIX_G, PIX_B]]], np.uint8)
_probe = np.tile(_p, (16, 16, 1))
_rng = np.random.default_rng(_s)
_probe = np.clip(_probe.astype(int) + _rng.integers(-60, 61, _probe.shape), 0, 255).astype(np.uint8)

_cg = cv2.cvtColor(_probe, cv2.COLOR_RGB2GRAY).astype(float)
_cy = cv2.cvtColor(_probe, cv2.COLOR_RGB2YCrCb).astype(float)
_ch = cv2.cvtColor(_probe, cv2.COLOR_RGB2HSV).astype(float)
_mh = rgb_to_hsv(_probe); _msk = _mh[..., 1] > .15
_dh = np.abs((_mh[..., 0] - _ch[..., 0]*2 + 180) % 360 - 180)
_checks = [("greyscale", float(np.abs(rgb_to_gray(_probe)-_cg).max()), 1.0),
           ("YCrCb order", float(np.abs(rgb_to_ycrcb(_probe)-_cy).max()), 2.0),
           ("hue units", float(_dh[_msk].max()) if _msk.any() else 0., 2.5),
           ("saturation", float(np.abs(_mh[..., 1]-_ch[..., 1]/255).max()), .02),
           ("value", float(np.abs(_mh[..., 2]-_ch[..., 2]/255).max()), .02),
           ("inverse", float(np.abs(ycrcb_to_rgb(rgb_to_ycrcb(_probe))-_probe.astype(float)).max()), 2.0)]
for n, d, t in _checks:
    print(f"  {'OK ' if d <= t else 'NO '} {n:12s} max diff {d:8.4f}  (allowed {t})")
A2_ok = all(d <= t for _, d, t in _checks)
print(f"\nA2: {'PASS (1 mark)' if A2_ok else 'FAIL'}")
if not A2_ok:
    print("  hue alone failing -> you returned OpenCV's units instead of the lecture's")
    print("  YCrCb off by ~100 -> you stacked Cb before Cr")

## The tools you will use for the rest of the lab

The next cell defines the functions you will work with from here on. You do not have to
write them, but you do have to know what each one tells you, because every mark below
depends on reading them correctly.

| Function | What you give it | What it gives back |
|:--|:--|:--|
| `psnr(a, b)` | two pictures | one number in dB: how close they are. 100 means identical, above 40 you cannot see a difference, around 30 is visibly degraded, below 25 is obviously broken |
| `shrink_and_stretch(ch)` | one channel | that channel shrunk to half size and blown straight back up. Whatever was too fine to survive the shrink is gone |
| `shrink_cost(ch)` | one channel | how much the channel changed when it was shrunk and stretched back. It is a reading of how much fine detail that channel holds — worth comparing between two pictures rather than reading on its own |
| `jpeg_block_seams(im)` | a picture | JPEG works on 8x8 squares and leaves faint seams along that grid. This says how much stronger the seams are than ordinary edges: about 1 means no seams |
| `measure(im)` | one picture | plain statistics about that picture on its own |
| `compare(a, b)` | two pictures | `psnr` worked out separately for R, G, B, Y, Cr and Cb, plus how far the colours have rotated |
| `encode(im, "PNG")` and `decode(bytes)` | a picture / some bytes | make a file in memory and read it back, without writing to disk |
| `show((im, "title"), ...)` | pictures with titles | draw them side by side |

The two you will lean on hardest are `compare`, because it reports each channel
separately, and `shrink_cost`, because it says which channel was carrying the detail.

In [ ]:
# ── the tools described above (given - do not edit) ──
def psnr(a, b):
    """How close two images are, in dB (the measure from the demo).
    Higher = closer. Above ~40 you cannot see the difference; ~30 is visibly
    degraded; below ~25 is obviously broken. 100 means identical."""
    mse = float(np.mean((np.asarray(a, float) - np.asarray(b, float))**2))
    return 100. if mse <= 0 else float(10*np.log10(255.**2/mse))

def shrink_and_stretch(ch, factor=2):
    """Scale one channel down by `factor`, then scale it straight back up.
    Whatever detail was too fine to survive the shrink is gone for good."""
    ch = np.asarray(ch, float)
    up = np.repeat(np.repeat(ch[::factor, ::factor], factor, 0), factor, 1)
    return up[:ch.shape[0], :ch.shape[1]]

def shrink_cost(ch, factor=2):
    """How much a channel changes when you shrink and stretch it back.
    Small = that channel had little fine detail to lose."""
    return float(np.std(np.asarray(ch, float) - shrink_and_stretch(ch, factor)))

def jpeg_block_seams(im):
    """JPEG applies the DCT to 8x8 squares, so it leaves faint seams on an
    8-pixel grid. This counts how much stronger the seams are than ordinary edges."""
    g = rgb_to_gray(im); dv = np.abs(np.diff(g, axis=0)); dh = np.abs(np.diff(g, axis=1))
    ov = dv[7::8, :].mean(); fv = np.delete(dv, np.s_[7::8], 0).mean()
    oh = dh[:, 7::8].mean(); fh = np.delete(dh, np.s_[7::8], 1).mean()
    return float((ov/max(fv, 1e-9) + oh/max(fh, 1e-9))/2)

def measure(im):
    """Absolute statistics of one image."""
    a = np.asarray(im, np.uint8); y = rgb_to_ycrcb(a); v = rgb_to_hsv(a)
    return {"mean_R": float(a[..., 0].mean()), "mean_G": float(a[..., 1].mean()),
            "mean_B": float(a[..., 2].mean()), "mean_Y": float(y[..., 0].mean()),
            "mean_Cr": float(y[..., 1].mean()), "mean_Cb": float(y[..., 2].mean()),
            "mean_S": float(v[..., 1].mean()),
            "shrink_cost_Y": shrink_cost(y[..., 0]),
            "shrink_cost_Cr": shrink_cost(y[..., 1]),
            "shrink_cost_Cb": shrink_cost(y[..., 2]),
            "levels": int(len(np.unique(a[..., 0]))),
            "jpeg_block_seams": jpeg_block_seams(a)}

def hue_shift(x, z):
    hx, hz = rgb_to_hsv(x), rgb_to_hsv(z)
    m = (hx[..., 1] > .15) & (hz[..., 1] > .15)
    return 0. if not m.any() else float(np.median((hz[..., 0][m]-hx[..., 0][m]+180) % 360 - 180))

def compare(x, z):
    """Compare two images one channel at a time. ~100 dB means that channel
    was not touched at all."""
    x, z = np.asarray(x, np.uint8), np.asarray(z, np.uint8)
    yx, yz = rgb_to_ycrcb(x), rgb_to_ycrcb(z)
    out = {"psnr_overall": psnr(x, z)}
    for i, n in enumerate("RGB"): out["psnr_"+n] = psnr(x[..., i], z[..., i])
    for i, n in enumerate(["Y", "Cr", "Cb"]): out["psnr_"+n] = psnr(yx[..., i], yz[..., i])
    out["hue_shift"] = hue_shift(x, z)
    return out

def encode(a, fmt, **kw):
    b = io.BytesIO(); Image.fromarray(np.asarray(a, np.uint8)).save(b, format=fmt, **kw)
    return b.getvalue()

def decode(blob): return np.array(Image.open(io.BytesIO(blob)).convert("RGB"))

def show(*pairs, w=3.2):
    fig, ax = plt.subplots(1, len(pairs), figsize=(w*len(pairs), w+.4))
    ax = np.atleast_1d(ax)
    for a, (im, t) in zip(ax, pairs):
        a.imshow(np.asarray(im, np.uint8)); a.set_title(t, fontsize=9); a.axis("off")
    plt.tight_layout(); plt.show()
print("instruments ready")

## Your picture

Everything from here runs on one photograph of your own, so it needs some colour in it.
A photo of a white wall makes half the measurements below read zero and there is nothing
to diagnose.

Set `SOURCE` to one of three things:

- `"camera"` — the webcam, through Colab. It asks for permission, then shows a capture button.
- `"upload"` — a photo from your phone. Better quality, and worth the extra half minute.
- `"synth"` — builds a picture out of two colours, if neither of the above works.

The cell also defines two helpers you will need again in Part B:

- `to_working(picture)` shrinks the picture so its longest side is 256 pixels and trims it
  to a whole number of 8-pixel blocks. Small pictures keep the lab fast, and the 8-pixel
  trim matters because JPEG works in 8x8 squares.
- `synth(colour_a, colour_b, cell, size, pattern)` builds a picture from two colours you
  choose. `cell` is how many pixels wide each square or stripe is, and `pattern` is
  `"checker"`, `"stripes"`, or anything else for two flat halves.

The `image code` printed at the end is a short code worked out from the picture itself.
Two people who submit the same picture get the same code, so use your own.

In [ ]:
SOURCE = "upload"        # "camera" | "upload" | "synth"

def to_working(arr, max_side=256):
    a = np.asarray(arr)[:, :, :3]
    h, w = a.shape[:2]; sc = max_side/max(h, w)
    if sc < 1:
        a = np.array(Image.fromarray(a.astype(np.uint8)).resize(
            (max(8, int(round(w*sc))), max(8, int(round(h*sc)))), Image.LANCZOS))
    h, w = a.shape[:2]
    return np.ascontiguousarray(a[:h-h % 8, :w-w % 8].astype(np.uint8))

def synth(colour_a=(220, 30, 40), colour_b=(30, 180, 60), cell=8, size=192, pattern="checker"):
    """Build a test image out of two colours. cell = size of each square/stripe."""
    a, b = np.array(colour_a, np.uint8), np.array(colour_b, np.uint8)
    yy, xx = np.mgrid[0:size, 0:size]
    if pattern == "checker":  m = ((yy//cell) + (xx//cell)) % 2 == 0
    elif pattern == "stripes": m = (xx//cell) % 2 == 0
    else:                      m = (yy < size//2)          # two flat panels
    return np.where(m[..., None], a, b).astype(np.uint8)

def grab_photo():
    from IPython.display import display, Javascript
    from google.colab.output import eval_js
    display(Javascript("""
      async function takePhoto(q){const d=document.createElement('div');
      const c=document.createElement('button');c.textContent='CLICK TO CAPTURE';
      c.style.cssText='font-size:18px;padding:10px 18px;margin:8px;cursor:pointer';
      d.appendChild(c);const v=document.createElement('video');
      v.style.cssText='display:block;max-width:420px';
      const s=await navigator.mediaDevices.getUserMedia({video:true});
      document.body.appendChild(d);d.appendChild(v);v.srcObject=s;await v.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight,true);
      await new Promise(r=>c.onclick=r);const k=document.createElement('canvas');
      k.width=v.videoWidth;k.height=v.videoHeight;k.getContext('2d').drawImage(v,0,0);
      s.getVideoTracks()[0].stop();d.remove();return k.toDataURL('image/jpeg',q);}"""))
    d = eval_js("takePhoto(0.9)")
    return np.array(Image.open(io.BytesIO(base64.b64decode(d.split(',')[1]))).convert("RGB"))

def load_image(source):
    if source == "camera":
        return grab_photo()
    if source == "upload":
        from google.colab import files
        up = files.upload()
        return np.array(Image.open(io.BytesIO(next(iter(up.values())))).convert("RGB"))
    return synth()

img = to_working(load_image(SOURCE))
show((img, f"your image {img.shape[1]}x{img.shape[0]}"))
print("image code:", hashlib.sha256(img.tobytes()).hexdigest()[:16])

## A3 · Work out what was done to your picture — 2 marks

The cell below takes **your** picture and puts it through **exactly one** of these ten
changes, chosen by your seed:

| | | |
|:--|:--|:--|
| `red_and_blue_swapped` | `colour_channels_swapped` | `colour_shrunk` |
| `brightness_shrunk` | `hue_shifted` | `colour_faded` |
| `few_levels` | `colour_removed` | `brightness_curved` |
| `low_quality_jpeg` | | |

Several of them look almost the same by eye. `colour_shrunk` and `colour_faded` both wash
the colour out. `brightness_shrunk` and `low_quality_jpeg` both go blocky. **Looking will
not settle it. Measuring will.**

### What each reading means

`compare(original, mystery)` puts the two pictures side by side one channel at a time:

| Reading | What it tells you |
|:--|:--|
| `psnr_overall` | how close the two pictures are altogether |
| `psnr_R`, `psnr_G`, `psnr_B` | the same, for the red, green and blue channels separately |
| `psnr_Y` | the same, for brightness on its own |
| `psnr_Cr`, `psnr_Cb` | the same, for the two colour channels on their own |
| `hue_shift` | how far the colours have been rotated round the colour wheel, in degrees. 0 means not rotated |

**Anything close to 100 here means that channel came through untouched,** and that one
fact rules out most of the list immediately.

`measure()` describes a single picture on its own. It is printed for the original and for
the mystery side by side, so you can see what moved:

| Reading | What it tells you |
|:--|:--|
| `mean_R`, `mean_G`, `mean_B` | the average red, green and blue value |
| `mean_Y` | the average brightness |
| `mean_Cr`, `mean_Cb` | how far the picture leans red, and how far it leans blue, on average. 128 means no lean either way |
| `mean_S` | how strong the colour is on average. 0 means completely grey |
| `shrink_cost_Y` | how much fine detail the brightness channel is carrying. Read it as original *against* mystery: if the number moved either way, the fine structure of that channel was interfered with |
| `shrink_cost_Cr`, `shrink_cost_Cb` | the same, for the two colour channels |
| `levels` | how many different values the red channel uses. An ordinary photo uses close to all 256 |
| `jpeg_block_seams` | how strong the 8-pixel seams are. About 1 means none; well above 1 means the picture has been through JPEG at a low quality |

### What you hand in

Fill in `DIAGNOSIS` below with the name, the readings you are relying on, and one sentence
saying why they settle it.

- **1 mark** for the right name together with at least one supporting number that is
  actually correct.
- **1 mark** if the number you lead with is one that separates your answer from its
  look-alike. Everyone can quote `psnr_overall`; it tells nobody which change it was.

A name on its own scores nothing.

In [ ]:
# The mystery generator. Reading this cell is not the exercise - measuring is.
exec(zlib.decompress(base64.b85decode(
    "c-o~_UvHy05P$EdKuBHLi9><>X*bAy*wty()$QHt-YSHU0RwIV<P1qeAMUf?*g%qQ8eOYH0z5PJ%=kCs8DEKFQ_03Cx5cJ4SZ##mh8UvKLs1bUVl`jPG#M>sSE;ND<LATgM!)W%U>VoahvF*yVNk<2B~L<DCSkPR^00npn+;D)V#d|F+R8A_Sy}RReJ-hUxh+>Fxia))Mp9&XQ}Vh#h|FB6%thLqE=*aX$&^2bYrX^JM<x9f$%|^IC!J`Gx3c4{tLO0VEnADG4WBkVGlQ#yr$*Q!hL>6sqX3M>Y9=KEE=kuFYp`cLQp^4PdBl9nvWOjwt@+(;PdPw2&a9{0K<kmb*~(I(Y}E~mSIB=Pj(az@Zk_1}4QeKjxZ}<jvw`a>Za^ju?tGJ|?R|SBUnTh((Y9Yr+TMOYxNLgtSTh^!ew7n2hsomv{Foi+i8Y-mD(a;${q}ZxV4OIU{|8Vv?tE@}l4K;J!zRf96M2PDyeDx<nN_eo0{sL=qEsU<q49namtL+-yl`37;aaRXiuc$nXmW3B1wzu|6YwaN*b~`6WcbrRLV6+^7?Ebco9SiGqRBNlTuQ*F;YgG_NQs1+A}cG&O_GS6Nce&7cdVY=7c!s=dxQ|mCJ|QTO`^W|dVxI<XM;GJNUTUD&H_AkwKy8Ya*|<1Dscqj8NLv_oG9Ws=TZT1Z<FLkF#tAEYztr!heMb!u`!AFr?HY~*Snt6T_23=>B9DE4Q#v?8?+l@V-r9gaQIc^cA_q16ef&nZ%^Y9awa+fq8c`n-z;nkU%MELs}RAoN$jmC2ta}%O`3M2fNlUL@8vA6o%)9QC?<RIf(e4;(Ek_s@AV%zn*XBzAO(JxNfq@qUVP2t2Kg&Wg^^a$5QZoX$;ynQ1thWG_#*J1f;VJX^PY~pDLy+(8WVI3Or?+>7aF8Ls*(qI1j!Fz!MBe<U86WBX&;0*e|2P(q$@64&6j+`S%Xf7aqcgdDS<a2sS0m81fq9q>o4Qns&lsCKAf{7;RJnQD_}Qk4#%@(1t$s5E}ULPZ6Q2&oqmxTY5}Eh*ZAe&KzE>*%ZU?MakXjDugfp>g5!y5sjj}>8PuxqY4tup3K<&U9@`<CF8chs&Aef1sGC$p=ukwb2Z?(sah2qt<1H`W?bvyvyXc&K>Vn-l_rGkPHq@il9n~e>Glu#=(J&gTY|-W7U|3g__rl*b>x-tGU<567{!!&8t^bgP2r(!#Gdx<%##0w};Cw7shHJyo-Ry%6|Lx+P1*55|6Cd*F{N{4ps_L(BQ_y`Dk4CQbuKh+7?s(Rhw&C@|p9o?Aw>V2(@%r<Rt(xsPiU^d!f;H6q{`0qQCh4zAbbi*Gx^+rRwc#Z?)gD<HZ=eBeIl>s6tI8dh4Kn}w_Qk|qVJz5+hecaAT((FQ88K=ObtdQgzC6_wd`AP5IE&W~^i@r*41f6bwF3uaW{(l7@9&`#Uc0spmi`4H7(Y$".encode())).decode())

mystery = make_mystery(img, SEED)
show((img, "original"), (mystery, "mystery"))

In [ ]:
# Take whatever readings you think will settle it.
_o, _m = measure(img), measure(mystery)
_c = compare(img, mystery)

print("compare(original, mystery) - anything near 100 was left alone")
for k, v in _c.items():
    print(f"   {k:14s} {v:9.2f}")
print("\nmeasure()                  original -> mystery")
for k in _o:
    print(f"   {k:14s} {_o[k]:9.2f} -> {_m[k]:9.2f}")

In [ ]:
DIAGNOSIS = {
    "name":    None,   # one of the ten, spelled exactly as it appears above
    "numbers": None,   # the readings you are relying on, as a dictionary,
                       #   for example {"psnr_Y": 48.7, "psnr_Cr": 33.8}
    "reason":  None,   # one sentence: why these readings rule out the look-alike
}
assert all(v is not None for v in DIAGNOSIS.values()), "Fill in all three."
print(json.dumps(DIAGNOSIS, indent=1, default=float))

---

# Part B · Find the pictures where the rules break — 4 marks

Two things you watched in the demo were true of that photograph. Neither of them is true
of every photograph. Here you go and find the pictures where each one comes out the other
way round.

In both challenges you must **write down what you are aiming for before you look at the
result.** That prediction carries half the mark, because it is the part that shows you
reasoned it out rather than tried things until something worked.

You may photograph something or build it with `synth(...)`. Building one is not the easy
way out — to build one on purpose you have to know exactly which property of a picture
decides the outcome.

## Challenge 1 · Make the colour channels matter more than the brightness one — 2 marks

In the demo, shrinking `Cr` and `Cb` to half size and stretching them back was invisible,
while doing exactly the same thing to `Y` was obvious — and that is despite shrinking two
channels rather than one, which throws away twice as much of the picture.

Find or build a picture where it comes out the other way round: the version with the
colour channels shrunk scores **worse** than the version with brightness shrunk.

Before you make it, answer this for yourself. If the brightness channel is normally the
one carrying the detail, what would a picture have to look like for its detail to sit in
the colour channels instead? Write that down as `PREDICT_1`.

In [ ]:
PREDICT_1 = None   # one sentence: what must be true of your picture, and why?
assert PREDICT_1, "Write your prediction before you make the image."

SOURCE_1 = "synth"       # "camera" | "upload" | "synth"
# Build or load your attempt. If you use synth(), pick the two colours deliberately.
img1 = to_working(load_image(SOURCE_1)) if SOURCE_1 != "synth" else to_working(
    synth(colour_a=(255, 0, 0), colour_b=(0, 255, 0), cell=4, pattern="checker"))

In [ ]:
def chroma_vs_luma(im):
    y = rgb_to_ycrcb(im)
    a = y.copy(); a[..., 1] = shrink_and_stretch(y[..., 1]); a[..., 2] = shrink_and_stretch(y[..., 2])
    b = y.copy(); b[..., 0] = shrink_and_stretch(y[..., 0])
    return (psnr(im, np.clip(ycrcb_to_rgb(a), 0, 255)),
            psnr(im, np.clip(ycrcb_to_rgb(b), 0, 255)))

P1_CHROMA, P1_LUMA = chroma_vs_luma(img1)
_m1 = measure(img1)
show((img1, "challenge 1 image"))
print(f"colour channels shrunk  (half the data kept):  {P1_CHROMA:6.2f} dB")
print(f"brightness channel shrunk (3/4 kept):           {P1_LUMA:6.2f} dB")
print(f"\nwhat each channel loses when shrunk:  "
      f"Y {_m1['shrink_cost_Y']:6.2f}   Cr {_m1['shrink_cost_Cr']:6.2f}   Cb {_m1['shrink_cost_Cb']:6.2f}")
C1_ok = P1_CHROMA < P1_LUMA - 0.5
print(f"\nCHALLENGE 1: {'BROKEN - rule reversed' if C1_ok else 'rule still holds, try again'}")

## Challenge 2 · Make PNG produce the smaller file — 2 marks

"JPEG is smaller" only means anything if the two files look about as good as each other.
JPEG at its lowest quality setting is always smaller and always looks terrible, so the
comparison has to hold quality fixed. The cell below does that: it finds the lowest JPEG
quality that still reaches 40 dB on your picture — the point at which you cannot see the
difference — and compares PNG's file size against JPEG at that setting.

Find or build a picture where **PNG comes out smaller.**

Think about what the two formats actually do. PNG predicts each pixel from its neighbours
and stores only the difference. JPEG cuts the picture into 8x8 squares and describes each
square as a blend of smooth patterns. Which kind of picture is easy for the first and hard
for the second? Write that down as `PREDICT_2`.

In [ ]:
PREDICT_2 = None   # one sentence: what must be true of this picture, and why?
assert PREDICT_2, "Prediction first."

SOURCE_2 = "synth"
img2 = to_working(load_image(SOURCE_2)) if SOURCE_2 != "synth" else to_working(
    synth(colour_a=(240, 240, 240), colour_b=(20, 20, 20), cell=24, pattern="stripes"))

In [ ]:
def png_vs_jpeg(im, target_db=40.0):
    q = next((q for q in range(1, 101)
              if psnr(im, decode(encode(im, "JPEG", quality=q))) >= target_db), 100)
    return {"q": q, "png": len(encode(im, "PNG", optimize=True)),
            "jpeg": len(encode(im, "JPEG", quality=q)),
            "jpeg_psnr": psnr(im, decode(encode(im, "JPEG", quality=q)))}

R2 = png_vs_jpeg(img2)
show((img2, "challenge 2 image"))
print(f"lowest JPEG quality reaching 40 dB : q = {R2['q']}  ({R2['jpeg_psnr']:.1f} dB)")
print(f"PNG  {R2['png']:7d} bytes")
print(f"JPEG {R2['jpeg']:7d} bytes")
C2_ok = R2["png"] < R2["jpeg"]
print(f"\nCHALLENGE 2: {'BROKEN - PNG wins' if C2_ok else 'JPEG still smaller, try again'}")

---

# Part C · Make the smallest file — 2 marks, and a leaderboard

Everyone works on the **same picture**: the class photo taken at the start of the session.

Write `compress`, which turns that picture into bytes, and `decompress`, which builds the
picture back from those bytes and nothing else. The smallest number of bytes wins, on one
condition: your rebuilt picture must still score at least **30 dB** against the original,
measured at the original size.

Anything you have met today is allowed — converting to another colour space, shrinking
channels, using fewer levels, making the picture smaller, any file format. Resizing is
allowed but you pay for it, because the comparison happens after it has been blown back up
to the original size.

You are given a plain JPEG as the starting point. Beat it.

In [ ]:
CLASS_URL = "https://raw.githubusercontent.com/Elakkiya16/BITS_F459_Computer_Vision_26_27/main/labs/lab02/class.png"
try:
    with urllib.request.urlopen(CLASS_URL, timeout=20) as r:
        target = to_working(np.array(Image.open(io.BytesIO(r.read())).convert("RGB")), 320)
    FILE_SOURCE = "class"
except Exception as e:
    print("could not fetch the class photo:", e, "\n-> falling back to your own image")
    target = to_working(img, 320); FILE_SOURCE = "fallback"
show((target, f"the target {target.shape[1]}x{target.shape[0]} ({FILE_SOURCE})"))
print("raw size:", target.nbytes, "bytes")

In [ ]:
def compress(im):
    """Turn the picture into bytes. Anything goes, as long as decompress() can undo it."""
    # The starting point: a plain JPEG, comfortably above 30 dB. Beat it.
    return encode(im, "JPEG", quality=60)

def decompress(packed, shape):
    """Build the picture back, at the given shape, from the packed bytes alone."""
    return decode(packed)

In [ ]:
_packed = compress(target)
_recon = decompress(_packed, target.shape)
FILE_BYTES, FILE_PSNR = len(_packed), psnr(target, _recon)
FILE_OK = FILE_PSNR >= 30.0
show((target, "original"), (_recon, f"yours - {FILE_PSNR:.2f} dB"))
print(f"your file: {FILE_BYTES} bytes   PSNR {FILE_PSNR:.2f} dB   "
      f"{'VALID' if FILE_OK else 'INVALID - under 30 dB, does not count'}")
print(f"compression ratio vs raw: {target.nbytes/max(FILE_BYTES,1):.1f}x")

---

# Part D · Write up what you found

Three or four sentences each, about **your own numbers** — not the general rule, but what
happened in your pictures and why. These are read and marked.

In [ ]:
EXPLAIN = {
    "why_colour_mattered_more": None,
        # Challenge 1: which channel ended up carrying your picture, and how do your three
        # shrink_cost numbers show it?

    "why_png_won": None,
        # Challenge 2: what is PNG doing well on your picture that JPEG is doing badly,
        # and why would the comparison be unfair without fixing the quality first?

    "smallest_file_reasoning": None,
        # Part C: what did you try, in what order, and which single change bought the most
        # bytes for the least damage?
}
assert all(v and len(str(v).strip()) > 40 for v in EXPLAIN.values()), "All three, properly."
for k, v in EXPLAIN.items(): print(f"{k}:\n  {v}\n")

In [ ]:
def _b64png(a):
    return base64.b64encode(encode(a, "PNG", optimize=True)).decode()

ANSWERS = {
    "lab": "lab02", "version": 2, "bits_id": BITS_ID.strip().upper(), "seed": SEED,
    "generated": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "A": {k: (str(v) if k == "max_channel" else float(v)) for k, v in A.items()},
    "A2_ok": bool(A2_ok),
    "diagnosis": DIAGNOSIS,
    "B": {"predict_1": PREDICT_1, "psnr_chroma": float(P1_CHROMA), "psnr_luma": float(P1_LUMA),
          "predict_2": PREDICT_2, "png_vs_jpeg": {k: float(v) for k, v in R2.items()}},
    "C": {"source": FILE_SOURCE, "bytes": int(FILE_BYTES), "psnr": float(FILE_PSNR),
          "valid": bool(FILE_OK), "file_b64": base64.b64encode(_packed).decode()},
    "EXPLAIN": EXPLAIN,
    "images": {"main": _b64png(img), "c1": _b64png(img1), "c2": _b64png(img2),
               "rebuilt": _b64png(_recon)},
}
print("===== LAB02 ANSWER BLOCK v2 =====")
print(json.dumps(ANSWERS, separators=(",", ":"), default=float))
print("===== END LAB02 ANSWER BLOCK =====")
print("\npackaged. Now save to GitHub.")

### Save it

**File → Save a copy in GitHub**

| Field | Value |
|:--|:--|
| Repository | `BITS-F459-Computer-Vision/f459-<your BITS ID, lowercase>` |
| Branch | `main` |
| File path | `lab02.ipynb` |
| Commit message | `Lab 2` |

Exactly `lab02.ipynb`, at the top level of your repo — no folder, no other name.

> **Run everything top to bottom, and run the answer-block cell last.** Colab saves the
> notebook with its outputs, and the outputs are what gets read. A notebook saved before
> running has nothing in it.

### Before you leave

- [ ] A1 shows PASS · A2 shows PASS
- [ ] `DIAGNOSIS` filled in with a name, the numbers, and a reason
- [ ] `PREDICT_1` written **before** challenge 1 ran; challenge 1 says BROKEN
- [ ] `PREDICT_2` written **before** challenge 2 ran; challenge 2 says BROKEN
- [ ] The smallest-file challenge reports VALID and is smaller than the plain JPEG
- [ ] All three `EXPLAIN` answers written
- [ ] Answer block run last, output visible
- [ ] Saved as `lab02.ipynb` in your repo, and you opened GitHub and checked it is there